# Previsão de RSSI por nó, compartilhada e conjunta

Notebook de inspeção do protocolo v2. Execute a otimização pelo comando `lora-benchmark --optimize`; esta análise carrega os resultados preservados.

Separação temporal, scalers de treino, janelas horárias completas e seleção apenas na validação. RSSI em dBm; erros em dB.


In [ ]:
import json
from pathlib import Path
from src.data_loader import get_prepared_datasets
from src.cli.run_benchmark import print_single_table


## 1. Carregamento e Inspeção dos Dados Sem Vazamento

In [ ]:
data = get_prepared_datasets(
    file_path='data/combined_hourly_data.csv',
    seq_length=24,
    pred_length=6
)
print(f"Treino: {data['train'][0].shape} -> {data['train'][1].shape}")
print(f"Validação: {data['val'][0].shape} -> {data['val'][1].shape}")
print(f"Teste: {data['test'][0].shape} -> {data['test'][1].shape}")
print('Nós monitorados:', data['target_names'])
print('Variáveis do modelo:', data['feature_names'])

## 2. Resultados do benchmark

Modelos: ARIMA, VARX, LSTMs dedicadas e compartilhada, Seq2Seq recorrente com atenção, NLinear, DLinear e STGNN.

NLinear e DLinear usam pesos temporais compartilhados sem mistura entre nós. Consulte README.md para as diferenças de entradas e arquiteturas.


In [ ]:
HORIZON = 6
OUTPUT_DIR = Path('benchmark_results/revised_gpu_seed42')
summary = OUTPUT_DIR / f'benchmark_summary_H{HORIZON}.json'
if not summary.exists():
    raise FileNotFoundError('Aguarde o benchmark ou selecione outro diretório com resultados completos.')
results = json.loads(summary.read_text())


## 3. Análise dos Resultados e Trade-off de Integração

In [ ]:
print_single_table(results, HORIZON)


## 4. Visualização dos Gráficos Gerados

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(OUTPUT_DIR / f'metrics_per_node_H{HORIZON}.png')))
display(Image(filename=str(OUTPUT_DIR / f'metrics_comparison_H{HORIZON}.png')))
display(Image(filename=str(OUTPUT_DIR / f'predictions_comparison_H{HORIZON}.png')))
